In [3]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import optuna
from xgboost import XGBRegressor
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Task:

Similar to Task 2A, apply two machine learning algorithms to your dataset, but now focus on
predicting a numerical target (i.e. a regression problem). For the basic dataset this means
your choice of two regression algorithms while for the advanced dataset one regression al-
gorithm which is inherently temporal and one which is not. Describe similar details as you
have for the classification problem. Highlight the differences you see between the two types
of prediction tasks

### Algorithm Choice

Non Temporal: XG Boost 

Temporal: LSTM

In [4]:
df_rf = pl.read_csv("data/rf_dataset.csv")
df_rf = df_rf.with_columns(pl.col("date").cast(pl.Date))


In [5]:
dates = df_rf.select(pl.col("date")).sort("date")
y = df_rf.select(pl.col("target"))
X = df_rf.sort(pl.col("date")).drop(["target", "id"])

#### Train Test Split:

In [7]:
unique_dates = df_rf.select("date").unique().sort("date")
cutoff_idx = int(len(unique_dates) * 0.8)
test_cutoff = unique_dates.item(cutoff_idx, 0)

In [ ]:
print(test_cutoff)

2014-05-25


In [ ]:
train_df = df_rf.filter(pl.col("date") < test_cutoff)
test_df = df_rf.filter(pl.col("date") >= test_cutoff)

X_train = train_df.drop(["target", "id", "date"])
y_train = train_df.select("target")
X_test = test_df.drop(["target", "id", "date"])
y_test = test_df.select("target")

## XG Boost

#### TS CV

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)
splits = list(tscv.split(X_train))

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    fold_scores = []
    for train_idx, val_idx in splits:
        model = XGBRegressor(**params, random_state=42)
        model.fit(
            X_train[train_idx.tolist()],
            y_train[train_idx.tolist()]
        )
        preds = model.predict(X_train[val_idx.tolist()])
        actual = y_train[val_idx.tolist()].to_numpy().flatten()
        fold_scores.append(mean_absolute_error(actual, preds))

    return np.mean(fold_scores)



In [ ]:
# Run optimization
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=100)

print("Best MAE:", study.best_value)
print("Best params:", study.best_params)

Best MAE: 0.47162969129650884
Best params: {'n_estimators': 212, 'max_depth': 3, 'learning_rate': 0.012262096647363743, 'subsample': 0.6014692685296253, 'colsample_bytree': 0.8108340495209071, 'reg_alpha': 6.24488985756167, 'reg_lambda': 0.12535318443059631}


### Retrain with best parameters:

In [ ]:
best_model = XGBRegressor(**study.best_params, random_state=42)
best_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8108340495209071
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import

### Final evaluation on test set


In [ ]:
test_preds = best_model.predict(X_test)
test_actual = y_test.to_numpy().flatten()
final_mae = mean_absolute_error(test_actual, test_preds)
final_mse = np.mean((test_actual - test_preds) ** 2)

print(f"Test MAE: {final_mae}")
print(f"Test MSE: {final_mse}")

Test MAE: 0.45786113958249136
Test MSE: 0.434926261721773


## LSTM

In [8]:
lstm_meta = pl.read_csv("data/lstm_meta.csv")
lstm_meta = lstm_meta.with_columns(pl.col("date").str.to_date())
X_lstm = np.load("data/lstm_X.npy")
y_lstm = np.load("data/lstm_y.npy")

train_mask = lstm_meta["date"] < test_cutoff
test_mask = lstm_meta["date"] >= test_cutoff

X_train = X_lstm[train_mask.to_numpy()]
X_test = X_lstm[test_mask.to_numpy()]
y_train = y_lstm[train_mask.to_numpy()]
y_test = y_lstm[test_mask.to_numpy()]

In [11]:
n_samples, n_steps, n_features = X_train.shape
X_train_flat = X_train.reshape(-1, n_features)
X_test_flat = X_test.reshape(-1, n_features)

mean = X_train_flat.mean(axis=0)
std = X_train_flat.std(axis=0)
std[std == 0] = 1 

# Normalise

X_train = ((X_train.reshape(-1, n_features) - mean) / std).reshape(n_samples, n_steps, n_features)
X_test = ((X_test.reshape(-1, n_features) - mean) / std).reshape(-1, n_steps, n_features)

In [12]:
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)
print(np.isnan(X_train).sum(), np.isnan(y_train).sum())

(1046, 5, 17) float64
(1046,) float64
0 0


In [13]:
X_tensor = torch.from_numpy(X_train.astype(np.float32))
y_tensor = torch.from_numpy(y_train.astype(np.float32))
print(X_tensor.shape, y_tensor.shape)

torch.Size([1046, 5, 17]) torch.Size([1046])


In [16]:
class MoodDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32))
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(MoodDataset(X_train, y_train), batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(MoodDataset(X_test, y_test), batch_size=32, shuffle=False, num_workers=0)


In [19]:
class MoodLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, timesteps, features)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # h_n shape: (num_layers, batch, hidden_size)
        # take the last layer's hidden state
        last_hidden = h_n[-1]
        # map to single prediction
        output = self.fc(last_hidden)
        return output.squeeze(-1)


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MoodLSTM(
    input_size=n_features,
    hidden_size=64,
    num_layers=2,
    dropout=0.2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.L1Loss()  # MAE loss, matching  XGBoost metric

n_epochs = 100
best_test_mae = float("inf")

for epoch in range(n_epochs):
    # Train
    model.train()
    train_losses = []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # Evaluate every 10 epochs
    if (epoch + 1) % 10 == 0:
        model.eval()
        all_preds = []
        all_actual = []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                preds = model(X_batch)
                all_preds.extend(preds.cpu().numpy())
                all_actual.extend(y_batch.numpy())

        test_mae = mean_absolute_error(all_actual, all_preds)
        test_mse = np.mean((np.array(all_actual) - np.array(all_preds)) ** 2)
        print(f"Epoch {epoch+1:3d} | Train MAE: {np.mean(train_losses):.4f} | Test MAE: {test_mae:.4f} | Test MSE: {test_mse:.4f}")

        if test_mae < best_test_mae:
            best_test_mae = test_mae
            torch.save(model.state_dict(), "best_lstm.pt")

Epoch  10 | Train MAE: 0.4909 | Test MAE: 0.5388 | Test MSE: 0.5319
Epoch  20 | Train MAE: 0.4263 | Test MAE: 0.5002 | Test MSE: 0.4616
Epoch  30 | Train MAE: 0.3957 | Test MAE: 0.5208 | Test MSE: 0.4624
Epoch  40 | Train MAE: 0.3759 | Test MAE: 0.4897 | Test MSE: 0.4361
Epoch  50 | Train MAE: 0.3566 | Test MAE: 0.4840 | Test MSE: 0.4251
Epoch  60 | Train MAE: 0.3366 | Test MAE: 0.4948 | Test MSE: 0.4171
Epoch  70 | Train MAE: 0.3090 | Test MAE: 0.4934 | Test MSE: 0.4350
Epoch  80 | Train MAE: 0.3006 | Test MAE: 0.5537 | Test MSE: 0.4973
Epoch  90 | Train MAE: 0.2707 | Test MAE: 0.5033 | Test MSE: 0.4433
Epoch 100 | Train MAE: 0.2447 | Test MAE: 0.5275 | Test MSE: 0.4715


In [21]:
import optuna

def lstm_objective(trial):
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    n_epochs = trial.suggest_int("n_epochs", 50, 200, step=50)

    # Walk forward CV on training data only
    tscv = TimeSeriesSplit(n_splits=3)
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        # Split and normalize per fold
        X_tr = X_train[train_idx]
        X_val = X_train[val_idx]
        y_tr = y_train[train_idx]
        y_val = y_train[val_idx]

        # Normalize using fold training stats
        n_s, n_st, n_f = X_tr.shape
        tr_flat = X_tr.reshape(-1, n_f)
        fold_mean = tr_flat.mean(axis=0)
        fold_std = tr_flat.std(axis=0)
        fold_std[fold_std == 0] = 1

        X_tr = ((X_tr.reshape(-1, n_f) - fold_mean) / fold_std).reshape(n_s, n_st, n_f)
        X_val = ((X_val.reshape(-1, n_f) - fold_mean) / fold_std).reshape(-1, n_st, n_f)

        # Loaders
        tr_loader = DataLoader(
            MoodDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, num_workers=0
        )
        val_loader = DataLoader(
            MoodDataset(X_val, y_val), batch_size=batch_size, shuffle=False, num_workers=0
        )

        # Model
        model = MoodLSTM(
            input_size=n_f,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout
        ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.L1Loss()

        # Train
        best_val_mae = float("inf")
        patience_counter = 0
        for epoch in range(n_epochs):
            model.train()
            for X_b, y_b in tr_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                optimizer.zero_grad()
                loss = criterion(model(X_b), y_b)
                loss.backward()
                optimizer.step()

            # Early stopping check every 10 epochs
            if (epoch + 1) % 10 == 0:
                model.eval()
                preds, actuals = [], []
                with torch.no_grad():
                    for X_b, y_b in val_loader:
                        preds.extend(model(X_b.to(device)).cpu().numpy())
                        actuals.extend(y_b.numpy())
                val_mae = mean_absolute_error(actuals, preds)
                if val_mae < best_val_mae:
                    best_val_mae = val_mae
                    patience_counter = 0
                else:
                    patience_counter += 1
                if patience_counter >= 3:
                    break

        fold_scores.append(best_val_mae)

    return np.mean(fold_scores)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="minimize")
study.optimize(lstm_objective, n_trials=50)

print(f"Best MAE: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

Best MAE: 0.4665
Best params: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.46304598972422367, 'lr': 0.0009245195294264142, 'batch_size': 64, 'n_epochs': 100}


In [22]:
# Normalize full train/test
n_s, n_st, n_f = X_train.shape
tr_flat = X_train.reshape(-1, n_f)
fold_mean = tr_flat.mean(axis=0)
fold_std = tr_flat.std(axis=0)
fold_std[fold_std == 0] = 1

X_tr_norm = ((X_train.reshape(-1, n_f) - fold_mean) / fold_std).reshape(n_s, n_st, n_f)
X_te_norm = ((X_test.reshape(-1, n_f) - fold_mean) / fold_std).reshape(-1, n_st, n_f)

# Loaders
best_p = study.best_params
train_loader = DataLoader(
    MoodDataset(X_tr_norm, y_train), batch_size=best_p["batch_size"], shuffle=True, num_workers=0
)
test_loader = DataLoader(
    MoodDataset(X_te_norm, y_test), batch_size=best_p["batch_size"], shuffle=False, num_workers=0
)

# Train final model
model = MoodLSTM(
    input_size=n_f,
    hidden_size=best_p["hidden_size"],
    num_layers=best_p["num_layers"],
    dropout=best_p["dropout"]
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=best_p["lr"])
criterion = nn.L1Loss()

best_test_mae = float("inf")
for epoch in range(best_p["n_epochs"]):
    model.train()
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        preds, actuals = [], []
        with torch.no_grad():
            for X_b, y_b in test_loader:
                preds.extend(model(X_b.to(device)).cpu().numpy())
                actuals.extend(y_b.numpy())
        test_mae = mean_absolute_error(actuals, preds)
        test_mse = np.mean((np.array(actuals) - np.array(preds)) ** 2)
        print(f"Epoch {epoch+1:3d} | Test MAE: {test_mae:.4f} | Test MSE: {test_mse:.4f}")
        if test_mae < best_test_mae:
            best_test_mae = test_mae
            torch.save(model.state_dict(), "best_lstm_tuned.pt")

# Final eval
model.load_state_dict(torch.load("best_lstm_tuned.pt"))
model.eval()
preds, actuals = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        preds.extend(model(X_b.to(device)).cpu().numpy())
        actuals.extend(y_b.numpy())
final_mae = mean_absolute_error(actuals, preds)
final_mse = np.mean((np.array(actuals) - np.array(preds)) ** 2)
print(f"\nFinal LSTM | Test MAE: {final_mae:.4f} | Test MSE: {final_mse:.4f}")

Epoch  10 | Test MAE: 0.5151 | Test MSE: 0.4636
Epoch  20 | Test MAE: 0.5146 | Test MSE: 0.4743
Epoch  30 | Test MAE: 0.5100 | Test MSE: 0.4736
Epoch  40 | Test MAE: 0.5018 | Test MSE: 0.4364
Epoch  50 | Test MAE: 0.5141 | Test MSE: 0.4394
Epoch  60 | Test MAE: 0.5330 | Test MSE: 0.4373
Epoch  70 | Test MAE: 0.5291 | Test MSE: 0.4487
Epoch  80 | Test MAE: 0.5527 | Test MSE: 0.4879
Epoch  90 | Test MAE: 0.5454 | Test MSE: 0.4760
Epoch 100 | Test MAE: 0.5710 | Test MSE: 0.5066

Final LSTM | Test MAE: 0.5018 | Test MSE: 0.4364
